# 08 — Plotly: Interactive Visualization
**Goal:** Build interactive charts whose hover, zoom, and click actions make
exploration natural — first with `plotly.express`, then with the more
configurable `plotly.graph_objects`.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print('plotly', px.__version__ if hasattr(px, '__version__') else 'installed')

## 1. The two APIs

- **`plotly.express` (`px`)** — high-level, one-call charts. Like seaborn but
  interactive.
- **`plotly.graph_objects` (`go`)** — low-level, full control. Like raw
  matplotlib.

Always start with `px`. Drop down to `go` only when you need custom traces,
animations, or layout tricks that `px` does not expose.

## 2. Your first interactive chart

`px.scatter` returns a `Figure`. In a notebook, the figure renders
interactively — hover, zoom, pan, lasso select, reset. In a script, call
`fig.write_html('out.html')` to share.

In [ ]:
np.random.seed(33)
df = pd.DataFrame({
    'spend' : np.random.gamma(2, 50, 200),
    'conv'  : np.random.gamma(1, 8, 200),
    'channel': np.random.choice(['Search', 'Social', 'Display', 'Email'], 200),
    'region' : np.random.choice(['EMEA', 'APAC', 'AMER'], 200),
})
fig = px.scatter(df, x='spend', y='conv', color='channel',
                  size='conv', hover_data=['region'],
                  title='Spend vs conversions (interactive)')
fig.show()

## 3. The `px` family at a glance

| `px` function | What it draws |
|---|---|
| `px.scatter` | Scatter (with optional size, symbol, facets) |
| `px.line` | Line, ordered by x (or by `line_group`) |
| `px.bar` | Bar (vertical) |
| `px.histogram` | Histogram |
| `px.box` / `px.violin` | Box / violin |
| `px.imshow` | Heatmap |
| `px.pie` / `px.sunburst` / `px.treemap` | Part-to-whole |
| `px.choropleth` / `px.scatter_geo` | Maps |
| `px.scatter_3d` / `px.line_3d` | 3D |
| `px.density_heatmap` / `px.density_contour` | 2D density |

In [ ]:
fig = px.histogram(df, x='conv', color='channel', nbins=30,
                   marginal='box', opacity=0.7,
                   title='Conversions distribution per channel')
fig.show()

In [ ]:
fig = px.box(df, x='channel', y='conv', color='region',
              title='Conversions by channel × region')
fig.show()

## 4. Faceting — `facet_row` and `facet_col`

Just like seaborn's `col=` / `row=`, plotly can facet by one or two variables.
Each facet shares the same axis ranges, which makes comparison easy.

In [ ]:
fig = px.scatter(df, x='spend', y='conv', color='channel',
                  facet_col='region', title='Spend vs conv — one panel per region')
fig.show()

## 5. Time series — `px.line` and date axes

Plotly handles date axes automatically. Use `range_x` or `range_y` to lock the
view to a window, `markers=True` to keep the underlying points visible.

In [ ]:
dates = pd.date_range('2024-01-01', periods=120, freq='D')
ts = pd.DataFrame({
    'date': np.tile(dates, 3),
    'channel': np.repeat(['Search', 'Social', 'Display'], 120),
    'value': np.concatenate([
        np.cumsum(np.random.normal(0.1, 1, 120)) + 10,
        np.cumsum(np.random.normal(0.2, 1.5, 120)) + 8,
        np.cumsum(np.random.normal(0.05, 0.7, 120)) + 15,
    ]),
})
fig = px.line(ts, x='date', y='value', color='channel',
               title='Three channels over time')
fig.show()

## 6. Customizing the layout

Every `Figure` has a `layout` you can mutate. Common knobs:

- `fig.update_layout(title=..., xaxis_title=..., yaxis_title=...)`
- `fig.update_traces(marker=dict(size=8))` — applies to all traces
- `fig.update_xaxes(tickformat='%b %Y', dtick='M2')` — date formatting

In [ ]:
fig = px.line(ts, x='date', y='value', color='channel')
fig.update_layout(
    title='Three channels — restyled',
    template='simple_white',
    xaxis_title='Month',
    yaxis_title='Value (k$)',
    legend=dict(orientation='h', y=-0.15),
    width=800, height=400,
)
fig.update_xaxes(tickformat='%b %Y', dtick='M1')
fig.show()

## 7. Dropping down to `go`

When you need a chart that `px` does not expose, build traces manually. The
pattern is: create a `go.Figure`, `add_trace` for each series, then
`update_layout`.

In [ ]:
fig = go.Figure()
for ch, sub in ts.groupby('channel'):
    fig.add_trace(go.Scatter(
        x=sub['date'], y=sub['value'],
        mode='lines+markers', name=ch,
        hovertemplate='%{x|%b %d}<br>%{y:.2f}<extra>'+ch+'</extra>',
    ))
fig.update_layout(
    title='go.Scatter — full control over hover and mode',
    template='simple_white', hovermode='x unified',
    width=800, height=400,
)
fig.show()

## 8. Subplots with `make_subplots`

Plotly's answer to GridSpec. You can mix plot types in the same figure.

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Spend (hist)', 'Conv vs spend (scatter)'))
fig.add_trace(go.Histogram(x=df['spend'], nbinsx=25, name='spend'), 1, 1)
fig.add_trace(go.Scatter(x=df['spend'], y=df['conv'], mode='markers',
                         name='points', marker=dict(size=6, opacity=0.5)), 1, 2)
fig.update_layout(template='simple_white', width=900, height=380, showlegend=False)
fig.show()

## 9. Animations

Plotly's killer feature: add `animation_frame=` to most `px` charts and you get
a slider + play button. Cheap, no JavaScript.

In [ ]:
ts2 = ts.copy()
ts2['month'] = ts2['date'].dt.to_period('M').astype(str)
fig = px.line(ts2, x='date', y='value', color='channel', animation_frame='month',
               title='Channel value by month (animated)')
fig.update_layout(template='simple_white', width=800, height=420)
fig.show()

## 10. Export — HTML, PNG, PDF

```python
fig.write_html('chart.html')          # interactive, no dependencies
fig.write_image('chart.png')          # static — needs kaleido
fig.write_image('chart.pdf')          # vector — needs kaleido
```

We added `kaleido` in `pyproject.toml` exactly for `write_image`.

In [ ]:
import os
os.makedirs('reports/dataviz', exist_ok=True)
small = px.line(ts.head(60), x='date', y='value', color='channel', title='Sample')
small.update_layout(template='simple_white')
small.write_html('reports/dataviz/sample.html')
print('wrote reports/dataviz/sample.html')

## Summary

| Tool | Use |
|---|---|
| `px` | One-call interactive charts, faceting, animation |
| `go` | Full control over traces, hover, layout |
| `make_subplots` | Mixed-type dashboards |
| `fig.update_layout` / `update_traces` | Restyling |
| `write_html` / `write_image` | Sharing |

**Next:** `09_subplots_layouts.ipynb` — composing multi-panel figures.